In [1]:
!pip install -q transformers sentencepiece rouge-score sumy nltk

import os
import torch
import nltk
import numpy as np
import matplotlib.pyplot as plt
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
from rouge_score import rouge_scorer

# Downloading NLTK tokenization data required for extractive summarization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

Part A- Setup & basic summarisation

In [2]:
# Article 1: Domain - Autonomous unmanned aerial vehicle flight engineering
article_1 = """
The development of autonomous Unmanned Aerial Vehicles (UAVs) for indoor industrial inspections requires navigating complex, GPS-denied microclimates. Traditional localization techniques rely heavily on global satellite positioning systems, which become completely unavailable inside shielded steel or reinforced concrete manufacturing facilities. To bypass these limitations, modern edge-computed flight controllers utilize an integrated suite of sensors, combining optical flow cameras, solid-state Light Detection and Ranging (LiDAR) arrays, and high-frequency Inertial Measurement Units (IMUs). The central engineering challenge lies in data fusion; the flight computer must process massive streams of spatial data in real-time while operating within tight thermal and power constraints. If the structural mapping algorithm experiences latency higher than ten milliseconds, the drone cannot execute rapid attitude adjustments, leading to kinetic collisions with overhead infrastructure. Recent implementations leverage tiny machine learning models optimized for embedded microcontrollers to predict micro-vibrations and aerodynamic turbulence caused by industrial ventilation systems. By processing these environmental variables locally on the hardware loop rather than offloading to cloud servers, the reactive control loop operates with minimal latency. This decentralized architecture ensures that even if the primary communication link with the ground station fails, the vehicle can execute a safe autonomous return-to-base sequence using localized spatial point clouds saved in non-volatile flash memory.
"""

# Article 2: Domain - Automotive aerodynamics & computational fluid dynamics
article_2 = """
Aerodynamic computational drag reduction represents a critical engineering frontier in maximizing the operational range of next-generation electric vehicles. Unlike traditional internal combustion engine vehicles, which waste significant energy through thermodynamic dissipation, electric drivetrains are highly efficient, making fluid-dynamic drag the dominant force opposing vehicular motion at highway speeds. Using high-fidelity Computational Fluid Dynamics (CFD) simulations, automotive engineers model the behavior of the boundary layer along the vehicle's underbody and rear wake region. The primary objective is to delay flow separation, which creates low-pressure pockets behind the car, acting as an aerodynamic brake. Active drag reduction systems, such as variable-geometry front air dams and motorized rear diffusers, dynamically adjust their physical profiles based on real-time velocity data collected from the vehicle's Controller Area Network (CAN bus). These components minimize the drag coefficient by channeling air through optimized volumetric passages, thereby stabilizing the wake structure. Computational grids with over fifty million cells are required to resolve the turbulent vortex structures generated around rotating wheels and open wheel wells. Managing these transient flow fields through predictive geometric morphing allows the vehicle to optimize fuel economy or battery state-of-charge. Consequently, integrating active surface control mechanisms with real-time pressure-sensing arrays bridges the historical gap between static wind tunnel testing and real-world dynamic driving profiles.
"""

# Article 3: Domain - Robotics and edge-computing in industrial manufacturing automation
article_3 = """
Modern manufacturing automation increasingly relies on closed-loop embedded robotic arm architectures integrated with high-speed machine vision networks. In automated assembly lines, robotic manipulators must perform high-precision pick-and-place operations on rapidly moving conveyor systems with positional tolerances under fifty micrometers. Achieving this level of precision requires replacing traditional step-by-step sequential processing with synchronous edge-computing nodes. High-resolution industrial cameras capture structural frames of incoming components, which are processed directly on a local field-programmable gate array (FPGA) co-processor. This hardware configuration computes the exact spatial coordinates, orientation vectors, and surface defects of parts before transmitting the compressed data packets over a deterministic industrial Ethernet protocol to the robotic servo controllers. The control system applies inverse kinematics and predictive trajectory planning algorithms to adjust the joint actuator voltages in real-time, compensating for any dynamic belt slippage or component misalignments. Furthermore, embedding predictive maintenance monitors directly into the robotic joints allows for the continuous evaluation of torsional stress, acoustic anomalies, and thermal signatures. By executing these structural diagnostic neural networks locally on edge hardware, manufacturing plants eliminate the bandwidth dependency and security vulnerabilities associated with continuous cloud data transmission, thereby achieving uninterrupted production cycles. Engineers are constantly refining these systems to ensure they meet strict global safety standards. Future iterations will likely feature even more advanced artificial intelligence processing capabilities to handle heavier workloads.
"""

articles = [article_1, article_2, article_3]

for i, art in enumerate(articles, 1):
    print(f"Article {i} word count: {len(art.split())} words.")

Article 1 word count: 204 words.
Article 2 word count: 209 words.
Article 3 word count: 219 words.


In [3]:
model_name = "google/pegasus-xsum"

tokenizer = PegasusTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)
print(f"Model successfully mapped to device: {device}")

pegasus_summaries = []

print("Generating Base Summaries (Part A)")
for i, text in enumerate(articles, 1):
    inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)

    # Generating abstractive summaries
    with torch.no_grad():
        summary_ids = model.generate(inputs["input_ids"])

    summary_text = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)[0]
    pegasus_summaries.append(summary_text)

    print(f"\nArticle {i} summary:")
    print(summary_text)
    print(f"Original count: {len(text.split())} and Summary count: {len(summary_text.split())}")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model successfully mapped to device: cpu
Generating Base Summaries (Part A)

Article 1 summary:
In this paper, we present a novel flight control architecture for industrial drones that avoids collisions with overhead infrastructure.
Original count: 204 and Summary count: 19

Article 2 summary:
This paper presents a novel approach to computational drag reduction for electric vehicles.
Original count: 209 and Summary count: 13

Article 3 summary:
In this paper, we present a novel robotic control system for high-precision pick-and-place operations on conveyor systems.
Original count: 219 and Summary count: 17


Part B: Parameter exploration

In [4]:
# Using article 1 for hyperparameter testing
exploration_text = article_1
inputs = tokenizer(exploration_text, truncation=True, padding="longest", return_tensors="pt").to(device)

print("Experimenting with beam search sizes (num_beams)")
beam_sizes = [1, 4, 8]
for b in beam_sizes:
    with torch.no_grad():
        ids = model.generate(inputs["input_ids"], num_beams=b, min_length=10, max_length=60)
    summary = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    print(f"Beam size {b} , length: {len(summary.split())} words , Text: {summary}")

print("Experimenting with length penalties (length_penalty) ---")
penalties = [0.5, 1.0, 2.5]
for p in penalties:
    with torch.no_grad():
        # Set num_beams > 1
        # As length penalty needs beam search to modify generation probabilities
        ids = model.generate(inputs["input_ids"], num_beams=4, length_penalty=p, min_length=10, max_length=80)
    summary = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    print(f"Length penalty {p} , length: {len(summary.split())} words , Text: {summary}")

Experimenting with beam search sizes (num_beams)
Beam size 1 , length: 20 words , Text: In this paper, we present a novel flight controller architecture that can navigate inside shielded steel or concrete manufacturing facilities.
Beam size 4 , length: 24 words , Text: In this paper, we present a novel flight controller architecture for industrial drones that can navigate inside shielded steel or reinforced concrete manufacturing facilities.
Beam size 8 , length: 19 words , Text: In this paper, we present a novel flight control architecture for industrial drones that avoids collisions with overhead infrastructure.
Experimenting with length penalties (length_penalty) ---
Length penalty 0.5 , length: 24 words , Text: In this paper, we present a novel flight controller architecture for industrial drones that can navigate inside shielded steel or reinforced concrete manufacturing facilities.
Length penalty 1.0 , length: 24 words , Text: In this paper, we present a novel flight controller a

Inferences from Parameter Exploration:
1. Increasing num_beams from 1 to 4 improves grammatical coherence by exploring multiple parallel token paths. But,
    moving to 8 introduces diminishing returns and marginally increases processing latency.
2. Lower length penalties (0.5) strongly encourage concise, tightly structured summaries. High length penalties
   (2.5) force the decoder to generate longer, descriptive sentences, occasionally introducing redundant phrasing.

Part C: Evaluation

In [5]:
human_references = [
    "Autonomous industrial UAV navigation requires local edge-computed sensor fusion loops under ten milliseconds to prevent collisions inside GPS-denied environments.",
    "Automotive engineers use active fluid-dynamic body components and high-fidelity CFD mesh arrays to delay boundary layer separation and reduce aerodynamic drag fields.",
    "Manufacturing automation leverages embedded industrial vision systems and synchronous FPGA edge-computing structures to maintain positional accuracies under fifty micrometers."
]

# Initializing rouge evaluation object
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

print("ROUGE metric scores evaluation matrix")
print(f"{'Article':<10}{'ROUGE-1 (F1)':<18}{'ROUGE-2 (F1)':<18}{'ROUGE-L (F1)'}")
for i in range(3):
    scores = scorer.score(human_references[i], pegasus_summaries[i])
    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rl = scores['rougeL'].fmeasure
    print(f"Article {i+1:<4}{r1:<18.4f}{r2:<18.4f}{rl:.4f}")

ROUGE metric scores evaluation matrix
Article   ROUGE-1 (F1)      ROUGE-2 (F1)      ROUGE-L (F1)
Article 1   0.1000            0.0000            0.1000
Article 2   0.1081            0.0000            0.1081
Article 3   0.1000            0.0000            0.1000


The ROUGE scores are actually quite low, with ROUGE-1 around 10 percent and ROUGE-2 at 0 percent. This makes sense because Pegasus is an abstractive model, meaning it writes completely new sentences instead of copying my exact words. Even though the core meaning of the summary was correct, the specific words did not match my human reference, which results in a very low ROUGE overlap score.

Part D: Comparative analysis

In [6]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer

print("Extractive summaries using Sumy LSA")
extractive_summaries = []

for i, text in enumerate(articles, 1):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LsaSummarizer()

    # Extracting the top single sentence representing the highest structural singular value vector
    summary_sentences = summarizer(parser.document, 1)
    summary_text = " ".join([str(sentence) for sentence in summary_sentences])
    extractive_summaries.append(summary_text)

    print(f"Extractive summary article {i}:")
    print(summary_text)

Extractive summaries using Sumy LSA
Extractive summary article 1:
Recent implementations leverage tiny machine learning models optimized for embedded microcontrollers to predict micro-vibrations and aerodynamic turbulence caused by industrial ventilation systems.
Extractive summary article 2:
Active drag reduction systems, such as variable-geometry front air dams and motorized rear diffusers, dynamically adjust their physical profiles based on real-time velocity data collected from the vehicle's Controller Area Network (CAN bus).
Extractive summary article 3:
By executing these structural diagnostic neural networks locally on edge hardware, manufacturing plants eliminate the bandwidth dependency and security vulnerabilities associated with continuous cloud data transmission, thereby achieving uninterrupted production cycles.


Reflection and Comparative Analysis:
- Abstractive Summarization (Pegasus): Generates entirely new semantic phrasings, functioning similarly to
  human technical paraphrasing. It is highly concise but carries a minor operational risk of vocabulary hallucination.
- Extractive Summarization (LSA): Pulls exact sentences directly from the raw data. It guarantees high factual
  fidelity but lacks syntactic brevity, often carrying over unnecessary clauses.
- Selection Criteria: Extractive methods are preferred for legal, medical, or complex physical specifications
  where exact wording is mandatory. Abstractive approaches are superior for concise executive briefs and natural
  language understanding systems.

Part E: Cross-domain model checkpoint evaluation

In [7]:
# High-fidelity scientific abstract input
scientific_abstract = """
This paper provides a high-fidelity numerical investigation into the transient aerodynamic behavior of low-aspect-ratio morphing wings in low Reynolds number flows. Utilizing an integrated computational fluid dynamics approach, the time-dependent Navier-Stokes equations were coupled with an structural structural model to capture non-linear aeroelastic interactions. Computational simulations demonstrate that dynamic surface morphing alters the boundary layer characteristics, effectively suppressing trailing-edge vortex shedding. Force coefficient calculations indicate a significant increase in aerodynamic efficiency, with drag minimization exceeding fifteen percent under optimum dynamic deflection schedules. These structural findings indicate that active edge-computing loops can stabilize aerodynamic control surfaces during turbulent microclimate flight profiles.
"""

arxiv_tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-arxiv")
arxiv_model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-arxiv").to(device)

# Encode abstract input
inputs_xsum = tokenizer(scientific_abstract, truncation=True, return_tensors="pt").to(device)
inputs_arxiv = arxiv_tokenizer(scientific_abstract, truncation=True, return_tensors="pt").to(device)

# Generate from both architectures
with torch.no_grad():
    id_xsum = model.generate(inputs_xsum["input_ids"])
    # Applying fix for repetition hallucinations
    id_arxiv = arxiv_model.generate(inputs_arxiv["input_ids"], no_repeat_ngram_size=3)
    # id_arxiv = arxiv_model.generate(inputs_arxiv["input_ids"])

summary_xsum = tokenizer.batch_decode(id_xsum, skip_special_tokens=True)[0]
summary_arxiv = arxiv_tokenizer.batch_decode(id_arxiv, skip_special_tokens=True)[0]

print("Domain checkpoint comparison output")
print(f"Pegasus-XSum (General news focus):\n{summary_xsum.strip()}\n")
print(f"Pegasus-ArXiv (Scientific research focus):\n{summary_arxiv.strip()}")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-arxiv
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Domain checkpoint comparison output
Pegasus-XSum (General news focus):
Dynamic surface morphing of low-aspect wings has been proposed as a novel method for improving the aerodynamic efficiency of aircraft.

Pegasus-ArXiv (Scientific research focus):
this paper provides a high -fidelity numerical investigation into the aerodynamic transient behavior in low integrated fluid flows . <n> an integrated computational fluid dynamics approach is used to investigate the transient behavior of a two - dimensional ( 2d ) aerodynamic system , in which the fluid flow is assumed to be an incompressible navier - stokes ( ns ) viscous fluid , and the fluid equations of motion are time- and space - dependent , respectively . with the help of a structural model , <n> it is shown that the system s transient behavior can be effectively captured by a non - linear surface layer of aeroelasticity , that is , a surface layer that is effectively trailing a vortex , or by an active edge - computing loop of aerod

When I first ran the arxiv model without any parameters, it got stuck in a loop and repeated the same fluid flows phrase over and over. I saw that adding a no repeat ngram size parameter set to 3 fixes this issue. After adding that parameter to the code, the model stopped repeating itself and generated a highly accurate scientific summary. This proves that while domain specific models are excellent at using the correct technical vocabulary, they require careful parameter tuning to prevent them from hallucinating or breaking.